## Building RAG and GenAI Pipelines on Vertex AI with Modern LangChain

This tutorial covers how to orchestrate generative AI tasks using LangChain on Google Cloud Platform (GCP). We will use the latest components from langchain-google-vertexai, langchain-core, and standard document retrieval architectures.

### Step 1: Environments & Modern Dependencies

The landscape of generative AI libraries moves fast. We install langchain-google-vertexai for direct integration with Vertex AI services, and switch from legacy vector indexing tools to langchain-chroma for efficient local document storage.

- langchain-google-vertexai houses the native drivers for Gemini and Vertex Embeddings.
- langchain-chroma provides modern vector storage wrappers that comply with standard LangChain interfaces.
- beautifulsoup4 and pypdf are modern processing backends for loading data from web scrapers and documents.

In [1]:
%pip install -U --quiet \
    google-cloud-aiplatform \
    langchain \
    langchain-community \
    langchain-google-vertexai \
    langchain-chroma \
    pypdf \
    beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


### Step 2: Initialization and Safety Headers

Before making API calls, configure your environment variables. A common issue with web scrapers in automated notebooks is missing identity signatures, which we handle by injecting a USER_AGENT.

Importing granular components from langchain_core allows us to construct lightweight, highly explicit pipelines instead of using old monolith chains.

In [2]:
import os
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings, VertexAIModelGarden
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Explicit authentication project target if required by your cloud instance
# os.environ["GOOGLE_CLOUD_PROJECT"] = "YOUR_PROJECT_ID"

# Set a structured agent identity to bypass scrape security blocks
os.environ["USER_AGENT"] = "modern-vertexai-tutorial/2.0"


/var/tmp/ipykernel_17047/860526196.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


### Step 3: Document Summarization (Modern LCEL Architecture)

Legacy LangChain utilized load_summarize_chain, which operated like a "black box." Modern applications use explicit LangChain Expression Language (LCEL) chains by piping Prompt | LLM | OutputParser.

- We use **gemini-2.5-flash**, Google's recommended model for high-speed, text-dense workloads.
- The | operator compiles the steps sequentially. The data enters the prompt, flows into Gemini, and maps directly into a readable Python string via StrOutputParser().


In [ ]:
# 1. Fetch data from an external scientific journal landing page
loader = WebBaseLoader("https://pubmed.ncbi.nlm.nih.gov/37883540/")
docs = loader.load()
docs_context = "\n\n".join([doc.page_content for doc in docs])

# 2. Instantiate the next-generation foundational model on Vertex AI
llm = ChatVertexAI(model_name="gemini-2.5-flash")

# 3. Create a structural chat container layout
summary_prompt = ChatPromptTemplate.from_template(
    "Write a single paragraph summary of the following document:\n\n{text}"
)

# 4. Bind components together using the pipe (|) operator
summary_chain = summary_prompt | llm | StrOutputParser()

# 5. Invoke the chain explicitly
response = summary_chain.invoke({"text": docs_context})
print("Document Summary:\n", response)


### Step 4: Building a Retrievable Knowledge Base (RAG)

For large documents, passing raw text can exceed optimization limits or cause hallucinations. We split a target file into searchable pieces, create vector footprints using text-embedding-005, and save them into a vector store.
- **RecursiveCharacterTextSplitter** divides text elegantly, keeping sentences together. An overlap of 150 characters prevents losing contextual meaning between splits.
- **text-embedding-005** is Google's latest embedding model, yielding highly precise context math.
- **vectorstore.as_retriever(search_kwargs={"k": 3})** constructs an active interface that pulls the 3 most statistically relevant text blocks when passed a question.

In [ ]:
# Download the target scientific text using standard system utilities
#!curl -L -o target_paper.pdf https://nih.gov

# Load the file layout into raw python document structures
loader = PyPDFLoader("s41586-024-07159-9.pdf")
raw_documents = loader.load()

#  Generate database splits using standard recursive overlap criteria
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=150)
splits = text_splitter.split_documents(raw_documents)

#  Initialize the Vertex AI Embedding instance
embeddings = VertexAIEmbeddings(model_name="text-embedding-005")

#  Generate footprints using your sanitized database chunks
vectorstore = Chroma.from_documents(documents=[splits[0]], embedding=embeddings)

#  Build your active database retriever
batch_size = 20  
for i in range(1, len(splits), batch_size):
    chunk_batch = splits[i : i + batch_size]
    # Add text segments safely to your persistence vector layout
    vectorstore.add_documents(documents=chunk_batch)

# 7. Expose the finalized vector workspace configuration to your active retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
#retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # Lowering k to 3 here keeps your subsequent chat steps safe as well


### Step 5: Context-Driven Question Answering (RAG Execution)

Now, we build a pipeline that takes a user query, automatically queries the database, stuffs the context into the prompt, and passes it to the model.
-  RunnablePassthrough() acts as a split pipeline lane. It forwards the user's string question directly to the {question} prompt variable, while sending the same query through the retriever to populate the {context} variable.
- This dynamic loading step allows the final application to run efficiently without requiring hardcoded text arguments.


In [ ]:
# Utility to stringify chunks cleanly for context embedding
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Design an explicit prompt layout forcing strict adherence to the data source
rag_prompt = ChatPromptTemplate.from_template("""Instructions: Use the following pieces of retrieved context to answer the question.
If you don't know the answer, say that you don't know. Be professional, factual, and succinct.

Context:
{context}

Question: {question}

Answer:""")

# Map the data stream using RunnablePassthrough to preserve raw user questions
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Run the target contextual execution trace
query = "What evidence is there that toothed whales go through menopause?"
print(rag_chain.invoke(query))


### Step 6: Code Generation and Specialized Tasks

Gemini 2.5 Flash is highly optimized for structural tasks like code drafting. This block handles isolated text production tasks.
- We adjust **temperature=0.1** to reduce the model's creativity, ensuring stable, reproducible, and syntactically valid code blocks.
- Increasing **max_output_tokens** ensures longer boilerplate applications do not get cut off mid-response.

In [ ]:
code_llm = ChatVertexAI(model_name="gemini-2.5-flash", max_output_tokens=2000, temperature=0.1)
code_prompt = ChatPromptTemplate.from_template(
    "You are an expert developer. Provide clean, well-commented code for the following task:\n{question}"
)
code_chain = code_prompt | code_llm | StrOutputParser()

print(code_chain.invoke({"question": "Write a python function that checks if a string is a valid email address"}))
